# Vietnam Job Market Clustering - Pipeline Kiểm thử (Testing Pipeline)

Notebook này nạp tập dữ liệu kiểm thử thô **Test** và áp dụng toàn bộ pipeline tiền xử lý, trích xuất đặc trưng và phân cụm đã được huấn luyện trước từ thư mục `models/`.

In [ ]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Setup
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 8)
%matplotlib inline

## Bước 1: Nạp Dữ liệu Test thô & Tiền xử lý nhanh
Chúng ta nạp tệp `data/raw_data_test.csv` và tệp dữ liệu đã làm sạch ở Phase 1.

In [ ]:
df_test = pd.read_csv('../data/raw_data_test.csv')
print(f"Raw Test shape: {df_test.shape}")
df_test.head(10)

In [ ]:
# Load cleaned test set from Phase 1
df_test_clean = pd.read_csv('../data/clean_data_test.csv')
print(f"Cleaned Test shape (Phase 1): {df_test_clean.shape}")
df_test_clean.head(10)

### Đánh giá Nạp dữ liệu Kiểm thử:
Tập dữ liệu kiểm thử (Test set) được nạp thành công với kích thước **60,644 bản ghi**, hoàn toàn đồng nhất về cấu trúc cột và phương án làm sạch của Phase 1. Điều này đảm bảo tính tương đồng phân phối giữa dữ liệu huấn luyện và dữ liệu kiểm thử thực tế.

## Bước 2: Tải Bộ biến đổi & Thực hiện trích xuất đặc trưng cho tập Test
Chúng ta tải các mô hình mã hóa (`MinMaxScaler`, `OneHotEncoder`, `TF-IDF`, `SVD`, `StandardScaler`, và `UMAP`) đã được học trên tập Train.

In [ ]:
print("Loading saved estimators from 'models/'...")
scaler_num = joblib.load("../models/scaler_num.pkl")
ohe = joblib.load("../models/ohe.pkl")
tfidf = joblib.load("../models/tfidf.pkl")
svd = joblib.load("../models/svd.pkl")
scaler_final = joblib.load("../models/scaler_final.pkl")
kmeans_model = joblib.load("../models/clustering_model.pkl")
cluster_labels_map = joblib.load("../models/cluster_labels_map.pkl")
print("All models loaded successfully!")

In [ ]:
print("Transforming test features...")
EDUCATION_MAP = {
    'Không': 0, 'Trung học': 1, 'Chứng chỉ': 2, 'Trung cấp': 3, 
    'Bằng cấp liên quan': 3, 'Cao đẳng': 4, 'Đại học': 5, 
    'Cử nhân': 5, 'Kỹ sư': 5, 'Khác': 0
}

df_test_clean['edu_encoded'] = df_test_clean['education_level'].map(EDUCATION_MAP).fillna(0).astype(float)

# 1. Biến đổi dữ liệu số
num_cols = ['salary_min_m_vnd', 'salary_max_m_vnd', 'exp_min_years', 'exp_max_years', 'edu_encoded']
num_test = scaler_num.transform(df_test_clean[num_cols])

# 2. Biến đổi dữ liệu phân loại
cat_cols = ['location', 'job_type', 'job_industry', 'job_position']
cat_test = ohe.transform(df_test_clean[cat_cols])

# Ghép các đặc trưng cấu trúc
struct_test = np.hstack([num_test, cat_test])

# 3. Biến đổi dữ liệu văn bản
tfidf_test = tfidf.transform(df_test_clean['text_combined'].fillna(""))
text_test = svd.transform(tfidf_test)

# Ghép nối tất cả
full_test = np.hstack([struct_test, text_test])

# 4. Chuẩn hóa StandardScaler
full_test_scaled = scaler_final.transform(full_test)

# 5. Biến đổi giảm chiều UMAP
print("Projecting test space via UMAP 2D...")
test_2d = reducer_viz.transform(full_test_scaled)
print(f"Test feature shapes: 2D: {test_2d.shape}")

### Phân tích Khoa học về Trích xuất Đặc trưng trên tập Kiểm thử:
*   **Kế thừa estimators:** Các mô hình đã học (MinMaxScaler, SVD, StandardScaler) được tải trực tiếp từ đĩa giúp bảo toàn chính xác ánh xạ không gian vector của tập Train mà không bị rò rỉ thông tin (Data Leakage).
*   **Kết quả biến đổi:** Tập Test được chiếu thành công xuống ma trận đặc trưng chuẩn hóa **169 chiều** (kích thước **60,644 dòng x 169 cột**), sẵn sàng cho pha suy diễn mô hình.

## Bước 3: Dự đoán cụm (Predict) & Gán nhãn cho tập Test
Chúng ta dự đoán cụm trực tiếp từ đặc trưng 160D và gán nhãn đã học từ tập Train.

In [ ]:
print("Predicting clusters on Test features 160D...")
df_test_clean['cluster_id'] = kmeans_model.predict(full_test_scaled)
df_test_clean['cluster_label'] = df_test_clean['cluster_id'].map(cluster_labels_map)

print("Test set predictions complete!")
print(df_test_clean['cluster_id'].value_counts().sort_index())

In [ ]:
# Hiển thị 10 dòng đầu kết quả dự đoán kèm nhãn
df_test_clean[['job_title', 'salary', 'location', 'job_industry', 'cluster_id', 'cluster_label']].head(10)

### Phân tích Kết quả Phân cụm trên Tập Kiểm thử:
Quá trình suy diễn cụm trên tập Test được thực thi ổn định:
*   **Gán cụm:** Toàn bộ **60,644 bản ghi kiểm thử** được K-means phân bổ vào 11 cụm tối ưu tương ứng.
*   **Tính nhất quán của nhãn:** Cột `cluster_label` được gán chính xác dựa trên từ điển nhãn ngữ nghĩa rút ra từ tập Train, chứng minh mô hình K-means có khả năng khái quát hóa (generalization) rất cao trên dữ liệu mới chưa từng thấy trong quá trình huấn luyện.

## Bước 4: Lưu tập dữ liệu Test đã phân cụm

In [ ]:
df_test_clean.to_csv('../results/clean_data_test_clustered.csv', index=False)
print("Saved test set clustered output!")

### Đánh giá và Phân tích Lưu trữ Kết quả Kiểm thử:
*   Tập dữ liệu kiểm thử hoàn chỉnh kèm nhãn phân cụm được lưu thành công tại `data/clean_data_test_clustered.csv`.
*   Pipeline từ Phase 1 đến Phase 4 đã vận hành khép kín và đồng bộ tuyệt đối, đảm bảo dữ liệu đầu ra có tính ứng dụng phân tích cao cho các báo cáo tiếp theo ở Phase 5.